# 문항1 : 콘텐츠 기반 필터링

## 1. 장르 기반 추천

1. 데이터 준비 및 이해

- tmdb_5000_movies 데이터셋을 로드하고, 데이터의 구조와 각 컬럼의 의미를 확인하세요.

In [6]:
import numpy as np
import pandas as pd
import warnings; warnings.filterwarnings('ignore')

movies = pd.read_csv('./tmdb_5000_movies.csv')

print(movies.shape)
display(movies.head(2).T)

(4803, 20)


,0,1
budget,237000000,300000000
genres,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""..."
homepage,http://www.avatarmovie.com/,http://disney.go.com/disneypictures/pirates/
id,19995,285
keywords,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na..."
original_language,en,en
original_title,Avatar,Pirates of the Caribbean: At World's End
overview,"In the 22nd century, a paraplegic Marine is di...","Captain Barbossa, long believed to be dead, ha..."
popularity,150.437577,139.082615
production_companies,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""..."


2. 데이터 전처리

- genres와 keywords 칼럼의 데이터를 리스트 형태로 변환하세요. ※ eval()

In [7]:
movies['genres'] = movies['genres'].apply(eval)
movies['keywords'] = movies['keywords'].apply(eval)

print(type(movies['genres'].iloc[0]))

<class 'list'>


3. 피처 엔지니어링

- genres 칼럼에서 장르 이름만 추출하여 새로운 칼럼 genres_literal을 생성합니다.

- 이 때, 추출된 이름을 공백으로 연결하여 문자열로 변환해주세요.

In [8]:
movies['genres_literal'] = movies['genres'].apply(lambda x: ' '.join([i['name'] for i in x]))

display(movies[['genres', 'genres_literal']].head(2))

,genres,genres_literal
0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",Action Adventure Fantasy Science Fiction
1,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",Adventure Fantasy Action


4. 벡터화

- genres_literal 칼럼을 CountVectorizer를 활용하여 벡터화하세요.

- 벡터의 shape와 전체 데이터 개수, genres_literal을 공백으로 분리한 고유값의 개수와 비교해보세요.

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
tf = CountVectorizer(analyzer='word', ngram_range=(1, 2), min_df=0.0, stop_words='english')
tfidf_matrix = tf.fit_transform(movies['genres_literal'])
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 20631 stored elements and shape (4803, 276)>

In [13]:
print(tfidf_matrix.shape)
tfidf_matrix.shape[0]

(4803, 276)


4803

In [14]:
all_genres = []
for words in movies['genres_literal']:
    all_genres.extend(words.split())

unique_genres_count = len(set(all_genres))
print(unique_genres_count)

22


5. 유사도 계산

- 위에서 구한 벡터를 활용하여 영화 간 유사도를 계산해보세요.

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
cosine_sim

array([[1.        , 0.59628479, 0.4472136 , ..., 0.        , 0.        ,
        0.        ],
       [0.59628479, 1.        , 0.4       , ..., 0.        , 0.        ,
        0.        ],
       [0.4472136 , 0.4       , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(4803, 4803))

In [17]:
titles = movies['title']
indices = pd.Series(titles.index, index=titles)
indices['Avatar']

np.int64(0)

In [19]:
sim_scores = cosine_sim[indices['Avatar']]
sim_scores

array([1.        , 0.59628479, 0.4472136 , ..., 0.        , 0.        ,
       0.        ], shape=(4803,))

In [20]:
sim_scores = list(enumerate(cosine_sim[indices['Avatar']]))
sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
sim_scores[:3]

[(0, np.float64(1.0000000000000002)),
 (14, np.float64(1.0000000000000002)),
 (46, np.float64(1.0000000000000002))]

In [ ]:
top10 = sim_scores[1:11]
movie_idxs = [i for i, _ in top10]
titles[movie_idxs]

14                                   Man of Steel
46                     X-Men: Days of Future Past
813                                      Superman
870                                   Superman II
3494    Beastmaster 2: Through the Portal of Time
1296                                 Superman III
1652                         Dragonball Evolution
419                                        Jumper
420                   Hellboy II: The Golden Army
72                                  Suicide Squad
Name: title, dtype: str

## 2. corpus(장르+키워드) 기반 추천

In [22]:
movies['keywords_literal'] = movies['keywords'].apply(lambda x: ' '.join([i['name'] for i in x]))

display(movies[['keywords', 'keywords_literal']].head(2))

,keywords,keywords_literal
0,"[{'id': 1463, 'name': 'culture clash'}, {'id':...",culture clash future space war space colony so...
1,"[{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...",ocean drug abuse exotic island east india trad...


In [23]:
movies['corpus'] = pd.Series(movies[['genres_literal', 'keywords_literal']].fillna('').values.tolist()).str.join(' ')

In [24]:
movies['corpus']

0       Action Adventure Fantasy Science Fiction cultu...
1       Adventure Fantasy Action ocean drug abuse exot...
2       Action Adventure Crime spy based on novel secr...
3       Action Crime Drama Thriller dc comics crime fi...
4       Action Adventure Science Fiction based on nove...
                              ...                        
4798    Action Crime Thriller united states–mexico bar...
4799                                      Comedy Romance 
4800    Comedy Drama Romance TV Movie date love at fir...
4801                                                     
4802     Documentary obsession camcorder crush dream girl
Name: corpus, Length: 4803, dtype: object

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tf_corpus = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=0.0, stop_words='english')
tfidf_matrix_corpus = tf_corpus.fit_transform(movies['corpus'])
tfidf_matrix_corpus

cosine_sim_corpus = linear_kernel(tfidf_matrix_corpus, tfidf_matrix_corpus)
cosine_sim_corpus

titles = movies['title']
indices = pd.Series(titles.index, index=titles)

In [26]:
cosine_sim_corpus

array([[1.        , 0.02129353, 0.01384659, ..., 0.01078978, 0.        ,
        0.        ],
       [0.02129353, 1.        , 0.0078472 , ..., 0.00852953, 0.        ,
        0.        ],
       [0.01384659, 0.0078472 , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.01078978, 0.00852953, 0.        , ..., 1.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        1.        ]], shape=(4803, 4803))

In [32]:
def corpus_recommendations(title, top_n=10):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim_corpus[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    top_n_movies = sim_scores[1:top_n+1]
    movie_idxs = [i for i, _ in top_n_movies]
    
    return titles[movie_idxs]

corpus_recommendations('Avatar', top_n=15)

278                  Planet of the Apes
2403                             Aliens
838                              Alien³
3730                              Cargo
4332                     Silent Running
373                     Mission to Mars
47              Star Trek Into Darkness
541                             Soldier
3208    Star Wars: Clone Wars: Volume 1
3158                              Alien
2198                            Lockout
239                             Gravity
1531                          Moonraker
1951                         Space Dogs
1473               The Astronaut's Wife
Name: title, dtype: str

# 문항 2: 아이템 기반 최근접 이웃 협업 필터링

1. 데이터 준비 및 이해

- MovieLens 데이터셋을 로드하고, 영화와 사용자의 평점 데이터를 로드해보세요.

In [34]:
movies2 = pd.read_csv('./movielens/movies.csv')
ratings = pd.read_csv('./movielens/ratings.csv')

print(movies2.shape, ratings.shape)

(9742, 3) (100836, 4)


In [35]:
display(movies2.head(2).T)
display(ratings.head(2))

,0,1
movieId,1,2
title,Toy Story (1995),Jumanji (1995)
genres,Adventure|Animation|Children|Comedy|Fantasy,Adventure|Children|Fantasy


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247


2. 데이터 전처리

- 영화 - 사용자 평점 데이터를 movieId를 기준으로 합쳐주세요. ※ merge

- 위의 결과를 rating에 대한 userId, title로 이루어진 피벗 테이블로 변환하세요.
※ pivot_table

In [39]:
rating_movies = pd.merge(movies2[['movieId', 'title', 'genres']], ratings, on='movieId')
rating_movies.head(10)

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483
5,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,18,3.5,1455209816
6,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,19,4.0,965705637
7,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,21,3.5,1407618878
8,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,27,3.0,962685262
9,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,31,5.0,850466616


In [55]:
user_counts = ratings.groupby('userId').size()
active_users = user_counts[user_counts>=100].index
rating_movies_active = rating_movies[rating_movies['userId'].isin(active_users)]

In [56]:
print(rating_movies_active.shape)

(84313, 6)


In [57]:
ratings_matrix = rating_movies_active.pivot_table('rating', index='userId', columns='title')
ratings_matrix.fillna(0, inplace=True)
ratings_matrix.shape

(248, 9552)

3. 유사도 계산

- 위에서 구한 벡터를 활용하여 영화 간 유사도를 계산해보세요.

In [50]:
ratings_matrix_T = ratings_matrix.T
ratings_matrix_T.shape

(9552, 248)

In [51]:
item_sim = cosine_similarity(ratings_matrix_T, ratings_matrix_T)
item_sim_df = pd.DataFrame(item_sim,
                           index=ratings_matrix_T.index,
                           columns=ratings_matrix_T.index)
item_sim_df

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...All the Marbles (1981),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.000000,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.166951,0.000000,0.000000,...,0.000000,0.394611,0.543305,0.707107,0.0,0.000000,0.145575,0.327327,0.000000,0.0
'Hellboy': The Seeds of Creation (2004),0.000000,1.000000,0.707107,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
'Round Midnight (1986),0.000000,0.707107,1.000000,0.0,0.0,0.198787,0.000000,0.000000,0.000000,0.707107,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
'Til There Was You (1997),0.000000,0.000000,0.000000,1.0,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
'Tis the Season for Love (2015),0.000000,0.000000,0.000000,0.0,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),0.000000,0.000000,0.000000,0.0,0.0,0.253475,0.230556,0.123035,0.154005,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.0,1.000000,0.213965,0.000000,0.191289,0.0
xXx (2002),0.145575,0.000000,0.000000,0.0,0.0,0.105236,0.000000,0.277758,0.022614,0.000000,...,0.072787,0.281842,0.180781,0.257343,0.0,0.213965,1.000000,0.281933,0.110428,0.0
xXx: State of the Union (2005),0.327327,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.184761,0.000000,0.000000,...,0.000000,0.441320,0.177838,0.231455,0.0,0.000000,0.281933,1.000000,0.000000,0.0


In [52]:
item_sim_df.iloc[:3,:3]

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986)
title,,,
'71 (2014),1.0,0.000000,0.000000
'Hellboy': The Seeds of Creation (2004),0.0,1.000000,0.707107
'Round Midnight (1986),0.0,0.707107,1.000000


4. 추천 시스템 구현

- 특정 영화와 유사한 영화를 추천하는 함수를 구현해보세요.

- 추천하는 영화의 이름으로 top_n개 만큼 추천 받을 수 있게 해보세요.

In [53]:
def find_sim_movie_item(df, title, top_n=10):
    sim = df[title].drop(title, axis=0)
    return sim.sort_values(ascending=False)[:top_n]

In [54]:
find_sim_movie_item(item_sim_df, 'Toy Story (1995)')

title
Jurassic Park (1993)                                                              0.740089
Forrest Gump (1994)                                                               0.713871
Star Wars: Episode VI - Return of the Jedi (1983)                                 0.700309
Star Wars: Episode IV - A New Hope (1977)                                         0.697383
Shrek (2001)                                                                      0.695893
Star Wars: Episode V - The Empire Strikes Back (1980)                             0.687684
Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)    0.683450
Back to the Future (1985)                                                         0.682072
Toy Story 2 (1999)                                                                0.679502
Apollo 13 (1995)                                                                  0.678452
Name: Toy Story (1995), dtype: float64

5. 특정 유저에게 영화 추천하는 시스템 구현

- 앞에서 구한 피벗 테이블을 userId를 기준으로 코사인 유사도를 계산한 뒤, 특정 유저를 위한 추천시스템을 구현해주세요.

In [60]:
user_sim = cosine_similarity(ratings_matrix, ratings_matrix)
user_sim_df = pd.DataFrame(user_sim,
                           index=ratings_matrix.index,
                           columns=ratings_matrix.index)
user_sim_df

userId,1,4,6,7,10,15,17,18,19,20,...,600,601,602,603,604,605,606,607,608,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.194395,0.128152,0.158744,0.016875,0.160689,0.264358,0.214868,0.325376,0.160969,...,0.291272,0.080554,0.164455,0.221486,0.070669,0.153625,0.164191,0.269389,0.291097,0.145321
4,0.194395,1.000000,0.088491,0.115120,0.031163,0.071551,0.145058,0.123217,0.206053,0.113755,...,0.211921,0.085938,0.128273,0.307973,0.052985,0.084584,0.200395,0.131746,0.149858,0.107683
6,0.128152,0.088491,1.000000,0.075843,0.020385,0.067576,0.090682,0.115582,0.221849,0.085544,...,0.191163,0.021415,0.448927,0.098002,0.396582,0.104541,0.102123,0.162182,0.178809,0.052668
7,0.158744,0.115120,0.075843,1.000000,0.132099,0.230475,0.246502,0.272828,0.126968,0.172800,...,0.220400,0.206405,0.125182,0.103664,0.062025,0.219586,0.200035,0.186114,0.323541,0.193219
10,0.016875,0.031163,0.020385,0.132099,1.000000,0.108254,0.072358,0.115139,0.026800,0.080516,...,0.098249,0.187179,0.013318,0.025047,0.011598,0.104130,0.088963,0.010451,0.077424,0.121072
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
605,0.153625,0.084584,0.104541,0.219586,0.104130,0.131857,0.174219,0.161859,0.164536,0.161765,...,0.244703,0.118538,0.112144,0.075659,0.080753,1.000000,0.148141,0.118780,0.178142,0.119295
606,0.164191,0.200395,0.102123,0.200035,0.088963,0.176102,0.228150,0.244718,0.197557,0.133759,...,0.310161,0.178084,0.116534,0.300669,0.066032,0.148141,1.000000,0.153063,0.262558,0.201104
607,0.269389,0.131746,0.162182,0.186114,0.010451,0.172278,0.232015,0.191474,0.267631,0.076034,...,0.219444,0.092525,0.199910,0.203540,0.137834,0.118780,0.153063,1.000000,0.283081,0.139114


In [61]:
def recommend_movies_by_user_sim(df, user_id, top_n=10):
    sim_users = df[user_id].drop(user_id, axis=0)
    top_sim_user = sim_users.sort_values(ascending=False).index[0]
    
    neighbor_ratings = ratings_matrix.loc[top_sim_user]
    target_user_ratings = ratings_matrix.loc[user_id]
    already_watched = target_user_ratings[target_user_ratings > 0].index
    recommend_candidates = neighbor_ratings[(neighbor_ratings >= 4.0) & (~neighbor_ratings.index.isin(already_watched))]
    
    return recommend_candidates.sort_values(ascending=False)[:top_n]


recommend_movies_by_user_sim(user_sim_df, user_id=1, top_n=10)

title
2001: A Space Odyssey (1968)                    5.0
Aliens (1986)                                   5.0
Austin Powers: The Spy Who Shagged Me (1999)    5.0
Blade Runner (1982)                             5.0
Fear and Loathing in Las Vegas (1998)           5.0
Brazil (1985)                                   5.0
Casino (1995)                                   5.0
Commitments, The (1991)                         5.0
Crow, The (1994)                                5.0
Election (1999)                                 5.0
Name: 266, dtype: float64